# ClinVar — API & bulk

| Mode | Functions | What it returns |
|---|---|---|
| **NCBI E-utilities** | `query_variant`, `query_gene` | One ClinVar record by UID / VCV accession, or every UID for a gene |
| **Bulk VCF** | `download_vcf`, `vcf_to_df`, `simplify_annotations`, `df_to_bed`, `df_to_sites` | Whole-corpus variant table with simplified `CLNSIG` buckets |

Bulk path needs the `[clinvar]` extra (`genoray`, `pooch`).

## 1. E-utilities — per-variant / per-gene lookups

The client respects NCBI's 3 req/sec free-tier limit automatically
(`_NCBI_RATE_LIMIT_SLEEP_S = 0.34`).

In [1]:
from biodb import clinvar

In [2]:
v = clinvar.query_variant(12345)
{k: v.get(k) for k in ("accession", "title", "obj_type", "clinical_significance")}

{'accession': 'VCV000012345',
 'title': 'NM_001065.4(TNFRSF1A):c.295T>A (p.Cys99Ser)',
 'obj_type': 'single nucleotide variant',
 'clinical_significance': None}

In [3]:
v2 = clinvar.query_variant("VCV000012345")
v2["accession"] == "VCV000012345"

True

In [4]:
uids = clinvar.query_gene("BRCA1", retmax=5)
uids

['4850689', '4849514', '4848953', '4848411', '4845640']

## 2. Bulk VCF — simplify the `CLNSIG` long tail

The full ClinVar VCF is ~70 MB. `simplify_annotations` collapses the
~30-way `CLNSIG` long tail into two side-by-side buckets: a 6-class
(`CLNSIG_simple`) and a 4-class (`CLNSIG_super_simple`) taxonomy.

In [5]:
import pandas as pd

# Synthetic minimal example — matches what you'd see in vcf_to_df output:
df = pd.DataFrame(
    {
        "CLNSIG": [
            "Pathogenic",
            "Likely_pathogenic",
            "Uncertain_significance",
            "Benign|Likely_benign",
            "Conflicting_classifications_of_pathogenicity",
            "drug_response",
        ],
    }
)

simplified = clinvar.simplify_annotations(df, verbose=False)
simplified[["CLNSIG", "CLNSIG_simple", "CLNSIG_super_simple"]]

,CLNSIG,CLNSIG_simple,CLNSIG_super_simple
0,Pathogenic,path,path
1,Likely_pathogenic,likely_path,path
2,Uncertain_significance,other,other
3,Benign|Likely_benign,other,other
4,Conflicting_classifications_of_pathogenicity,conflicting,conflicting
5,drug_response,other,other


`download_vcf` + `vcf_to_df` pull the full release and shape it for
pandas. (Not executed — 70 MB download.)

```python
vcf_path = clinvar.download_vcf()
all_variants = clinvar.vcf_to_df(vcf_path)            # ~ 2 M rows
annotated = clinvar.simplify_annotations(all_variants)
bed = clinvar.df_to_bed(annotated)                    # → BED for IGV / bedtools
sites = clinvar.df_to_sites(annotated)                # → chr:pos sites for downstream tooling
```